# Qwen3.5-9B-Base 工具调用 SFT 微调

使用 Unsloth + TRL 对 Qwen3.5-9B-Base 进行监督微调，训练工具调用/Agent 能力。

**数据**: 100k 中英混合工具调用样本 (7 个公开数据集)

**GPU 自适应**: 自动检测 T4/L4/A100 并调整参数

---

## 运行方式
1. 点击 `运行时` → `更改运行时类型` → 选择 `T4 GPU` (免费) 或 `A100` (Pro)
2. 依次运行每个 Cell
3. 训练完成后模型自动保存到 Google Drive

## 1. 安装依赖

In [1]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install deepspeed

## 2. 检测 GPU 并设置训练参数

In [2]:
import torch

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

# 根据 GPU 自动调参
# Qwen3.5-9B 词表 248k，logits 矩阵大。用 CPU offload + grad_ckpt 省显存
if vram_gb >= 70:  # A100 80GB / H100
    CONFIG = {
        "load_in_4bit": False,
        "max_seq_length": 4096,
        "batch_size": 4,
        "grad_accum": 4,
        "lora_r": 32,
    }
    print("→ A100-80GB/H100: bf16 LoRA, batch=4, seq=4096")
elif vram_gb >= 35:  # A100 40GB
    CONFIG = {
        "load_in_4bit": False,  # bf16 精度更好
        "max_seq_length": 2048,
        "batch_size": 4,
        "grad_accum": 4,
        "lora_r": 16,           # rank 16 省 ~1GB 显存
    }
    print("→ A100-40GB: bf16 LoRA, batch=4, seq=2048, offload+unsloth_ckpt")
elif vram_gb >= 20:  # L4 24GB / A10
    CONFIG = {
        "load_in_4bit": True,
        "max_seq_length": 2048,
        "batch_size": 4,
        "grad_accum": 4,
        "lora_r": 16,
    }
    print("→ L4/A10: 4-bit QLoRA, batch=4, seq=2048")
else:  # T4 16GB
    CONFIG = {
        "load_in_4bit": True,
        "max_seq_length": 1024,
        "batch_size": 2,
        "grad_accum": 8,
        "lora_r": 8,
    }
    print("→ T4: 4-bit QLoRA, batch=2, seq=1024")

print(f"\n配置: {CONFIG}")

GPU: Tesla T4
VRAM: 15.6 GB
→ T4 模式: 4-bit QLoRA, batch=1, seq=2048

配置: {'load_in_4bit': True, 'max_seq_length': 2048, 'batch_size': 1, 'grad_accum': 16, 'lora_r': 16}


## 3. 加载模型

In [1]:
from unsloth import FastLanguageModel
import gc, torch

# 清理之前可能残留的模型显存
if 'model' in dir(): del model
if 'processor' in dir(): del processor
if 'tokenizer' in dir(): del tokenizer
gc.collect()
torch.cuda.empty_cache()

load_4bit = CONFIG["load_in_4bit"]
model, processor = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-9B-Base",
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=load_4bit,
    load_in_16bit=not load_4bit,
    dtype=None,
)

# Qwen3.5 是 VLM，Unsloth 返回 processor 而不是 tokenizer
# 需要提取内部的 tokenizer 才能用 apply_chat_template
if hasattr(processor, 'tokenizer'):
    tokenizer = processor.tokenizer
    print(f'从 processor 中提取 tokenizer: {type(tokenizer).__name__}')
else:
    tokenizer = processor

print(f"模型加载完成: {type(model).__name__}")
print(f"has chat_template: {tokenizer.chat_template is not None}")
print(f"VRAM 使用: {torch.cuda.memory_allocated()/1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


NameError: name 'CONFIG' is not defined

## 4. 配置 LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_r"] * 2,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可训练参数: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

## 5. 准备数据

两种方式选一种:
- **方式 A**: 从本地上传已准备好的 `train.jsonl` / `valid.jsonl`
- **方式 B**: 在 Colab 中重新从 HuggingFace 下载并转换 (需要 ~5 分钟)

In [ ]:
#@title 选择数据加载方式 { run: "auto" }
DATA_SOURCE = "download_fresh"  #@param ["upload_local", "download_fresh"]

import os
os.makedirs("/content/data", exist_ok=True)

In [ ]:
# === 方式 A: 上传本地文件 ===
# 如果你选了 upload_local，运行这个 cell 会弹出文件选择框
if DATA_SOURCE == "upload_local":
    from google.colab import files
    print("请上传 train.jsonl 和 valid.jsonl")
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(f"/content/data/{name}", "wb") as f:
            f.write(content)
        print(f"  已保存: /content/data/{name}")
else:
    print("跳过上传，将在下一步从 HuggingFace 下载数据")

In [ ]:
# === 方式 B: 从 HuggingFace 下载并转换 ===
if DATA_SOURCE == "download_fresh" and not os.path.exists("/content/data/train.jsonl"):
    import json, hashlib, random, re
    from datasets import load_dataset

    def wrap_tool_openai(tool_def):
        if "type" in tool_def and tool_def["type"] == "function" and "function" in tool_def:
            return tool_def
        func = {"name": tool_def.get("name", "unknown"), "description": tool_def.get("description", "")}
        params = tool_def.get("parameters", {})
        if isinstance(params, dict) and "properties" in params:
            func["parameters"] = params
        else:
            func["parameters"] = {"type": "object", "properties": {}}
        return {"type": "function", "function": func}

    def make_tool_call(name, arguments):
        if isinstance(arguments, str):
            try: arguments = json.loads(arguments)
            except: arguments = {"raw": arguments}
        if not isinstance(arguments, dict): arguments = {"value": arguments}
        return {"type": "function", "function": {"name": name, "arguments": arguments}}

    def convert_sharegpt(ds, lang="en"):
        """转换 ShareGPT 格式 (glaive_zh, glaive_v2_sharegpt)"""
        results = []
        sys_msg = "你是一个有用的助手，可以调用工具来帮助用户完成任务。" if lang == "zh" else "You are a helpful assistant with access to tools."
        for row in ds:
            try:
                convs = row.get("conversations", [])
                tools_str = row.get("tools", "[]")
                tools = [wrap_tool_openai(t) for t in (json.loads(tools_str) if isinstance(tools_str, str) else tools_str) if isinstance(t, dict)] if tools_str else []
                messages = []
                if tools: messages.append({"role": "system", "content": sys_msg})
                for conv in convs:
                    rf, val = conv.get("from", ""), conv.get("value", "")
                    if rf == "human": messages.append({"role": "user", "content": val})
                    elif rf == "gpt": messages.append({"role": "assistant", "content": val})
                    elif rf == "function_call":
                        try:
                            fc = json.loads(val) if isinstance(val, str) else val
                            messages.append({"role": "assistant", "content": None, "tool_calls": [make_tool_call(fc.get("name","unknown"), fc.get("arguments",{}))]})
                        except: messages.append({"role": "assistant", "content": val})
                    elif rf == "observation":
                        tn = ""
                        for m in reversed(messages):
                            if m.get("role")=="assistant" and m.get("tool_calls"): tn=m["tool_calls"][0]["function"]["name"]; break
                        messages.append({"role": "tool", "content": val if isinstance(val,str) else json.dumps(val,ensure_ascii=False), "name": tn})
                roles = [m["role"] for m in messages]
                if "user" in roles and "assistant" in roles:
                    sample = {"messages": messages}
                    if tools: sample["tools"] = tools
                    results.append(sample)
            except: continue
        return results

    all_samples = []

    # 1. Deepexi (中文, 24.6k)
    print("[1/7] Deepexi (中文)...")
    ds = load_dataset("Deepexi/function-calling-small", split="train")
    for row in ds:
        try:
            up, ar = row.get("userPrompt",""), row.get("assistantResponse","")
            if not up or not ar: continue
            resp = json.loads(ar)
            tc = []
            if isinstance(resp, dict) and "function" in resp:
                args = resp.get("arguments",{})
                if isinstance(args,list) and args: args = args[0] if isinstance(args[0],dict) else {}
                tc = [make_tool_call(resp["function"], args)]
            msgs = [{"role":"system","content":"你是一个有用的助手，可以调用工具来帮助用户完成任务。"},{"role":"user","content":up}]
            if tc: msgs.append({"role":"assistant","content":None,"tool_calls":tc})
            else: msgs.append({"role":"assistant","content":ar})
            all_samples.append({"messages":msgs})
        except: continue
    print(f"  {len(all_samples)} samples")

    # 2. glaive_zh (中文, 1k)
    print("[2/7] glaive_zh (中文)...")
    ds = load_dataset("llamafactory/glaive_toolcall_zh", split="train")
    r = convert_sharegpt(ds, "zh")
    all_samples.extend(r)
    print(f"  {len(r)} samples")

    # 3. glaive_v2_sharegpt (英文, 100k)
    print("[3/7] glaive_v2_sharegpt (英文)...")
    ds = load_dataset("hiyouga/glaive-function-calling-v2-sharegpt", split="train")
    r = convert_sharegpt(ds, "en")
    all_samples.extend(r)
    print(f"  {len(r)} samples")

    # 4. hermes_fc (英文, 1.9k)
    print("[4/7] hermes_fc (英文)...")
    ds = load_dataset("NousResearch/hermes-function-calling-v1", split="train")
    tc_pat = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.DOTALL)
    for row in ds:
        try:
            convs = row.get("conversations",[])
            tools_str = row.get("tools","[]")
            tools = [wrap_tool_openai(t) for t in json.loads(tools_str)] if tools_str else []
            msgs = []
            if tools: msgs.append({"role":"system","content":"You are a helpful assistant with access to tools."})
            for c in convs:
                rf, val = c.get("from",""), c.get("value","")
                if rf=="system": continue
                elif rf=="human": msgs.append({"role":"user","content":val})
                elif rf=="gpt":
                    matches = tc_pat.findall(val)
                    if matches:
                        tcs = []
                        for m in matches:
                            try: d=json.loads(m); tcs.append(make_tool_call(d.get("name","?"),d.get("arguments",{})))
                            except: pass
                        if tcs: msgs.append({"role":"assistant","content":None,"tool_calls":tcs})
                        else: msgs.append({"role":"assistant","content":val})
                    else: msgs.append({"role":"assistant","content":val})
            roles=[m["role"] for m in msgs]
            if "user" in roles and "assistant" in roles:
                s={"messages":msgs}
                if tools: s["tools"]=tools
                all_samples.append(s)
        except: continue
    print(f"  done")

    # 5. ToolACE-Qwen (英文, 10.5k)
    print("[5/7] ToolACE-Qwen (英文)...")
    ds = load_dataset("tryumanshow/ToolACE-Qwen-cleaned", split="train")
    for row in ds:
        try:
            tools = [wrap_tool_openai(t) for t in json.loads(row.get("tools","[]"))] if row.get("tools") else []
            convs = json.loads(row.get("conversations","[]")) if isinstance(row.get("conversations"),str) else row.get("conversations",[])
            msgs=[]
            if tools: msgs.append({"role":"system","content":"You are a helpful assistant with access to tools."})
            for c in convs:
                role,content = c.get("role",""), c.get("content","")
                if role=="user": msgs.append({"role":"user","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False)})
                elif role=="assistant":
                    tc_raw=c.get("tool_calls",[])
                    if tc_raw:
                        tcs=[]
                        for tc in tc_raw:
                            fd=tc.get("function",{}); args=fd.get("arguments",{})
                            if isinstance(args,str):
                                try: args=json.loads(args)
                                except: args={"raw":args}
                            tcs.append(make_tool_call(fd.get("name","?"),args))
                        msgs.append({"role":"assistant","content":None,"tool_calls":tcs})
                    else: msgs.append({"role":"assistant","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False)})
                elif role=="tool":
                    tn=c.get("name","")
                    if not tn:
                        for m in reversed(msgs):
                            if m.get("role")=="assistant" and m.get("tool_calls"): tn=m["tool_calls"][0]["function"]["name"]; break
                    msgs.append({"role":"tool","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False),"name":tn})
            roles=[m["role"] for m in msgs]
            if "user" in roles and "assistant" in roles:
                s={"messages":msgs}
                if tools: s["tools"]=tools
                all_samples.append(s)
        except: continue
    print(f"  done")

    # 6. Opus reasoning (英文, 2.3k)
    print("[6/7] Opus Reasoning (英文)...")
    ds = load_dataset("nohurry/Opus-4.6-Reasoning-3000x-filtered", split="train")
    for row in ds:
        p, t, s = row.get("problem",""), row.get("thinking",""), row.get("solution","")
        if p and s:
            ac = f"<think>\n{t}\n</think>\n\n{s}" if t else s
            all_samples.append({"messages":[{"role":"user","content":p},{"role":"assistant","content":ac}]})
    print(f"  done")

    # 7. OpenClaw (英文, 7.2k)
    print("[7/7] OpenClaw (英文)...")
    for split in ["train","test"]:
        try:
            ds = load_dataset("bellfire/openclaw-coder-dataset", split=split)
            for row in ds:
                raw = row.get("messages",[])
                if not raw: continue
                msgs=[]
                for m in raw:
                    role,content = m.get("role",""), m.get("content","")
                    if role in ("system","user"):
                        msgs.append({"role":role,"content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False)})
                    elif role=="assistant":
                        tc_raw=m.get("tool_calls",[])
                        if tc_raw:
                            tcs=[make_tool_call(tc.get("function",{}).get("name","?"),tc.get("function",{}).get("arguments",{})) for tc in tc_raw]
                            msgs.append({"role":"assistant","content":None,"tool_calls":tcs})
                        else: msgs.append({"role":"assistant","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False)})
                    elif role=="tool":
                        tn=m.get("name","")
                        msgs.append({"role":"tool","content":content if isinstance(content,str) else json.dumps(content,ensure_ascii=False),"name":tn})
                roles=[m["role"] for m in msgs]
                if "user" in roles and "assistant" in roles:
                    all_samples.append({"messages":msgs})
        except: pass
    print(f"  done")

    # 去重 + 打乱 + 划分
    print(f"\n总计: {len(all_samples)} 样本")
    seen = set()
    deduped = []
    for s in all_samples:
        h = hashlib.md5(json.dumps(s["messages"],sort_keys=True,ensure_ascii=False).encode()).hexdigest()
        if h not in seen: seen.add(h); deduped.append(s)
    print(f"去重后: {len(deduped)} 样本")

    random.seed(42)
    random.shuffle(deduped)
    split_idx = int(len(deduped) * 0.9)

    for name, data in [("train.jsonl", deduped[:split_idx]), ("valid.jsonl", deduped[split_idx:])]:
        with open(f"/content/data/{name}", "w") as f:
            for s in data: f.write(json.dumps(s, ensure_ascii=False) + "\n")
        print(f"  {name}: {len(data)} 样本")

    print("数据准备完成!")
else:
    print("数据已存在，跳过")

In [ ]:
# 验证数据格式
import json

with open("/content/data/train.jsonl") as f:
    sample = json.loads(f.readline())

print(f"样本字段: {list(sample.keys())}")
print(f"消息轮次: {len(sample['messages'])}")
print(f"有 tools: {'tools' in sample}")

# 测试 apply_chat_template
try:
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tools=sample.get("tools"),
        tokenize=False,
    )
    print(f"\nTokenized 长度: {len(tokenizer.encode(text))} tokens")
    print(f"前 300 字符:\n{text[:300]}")
except Exception as e:
    print(f"ERROR: {e}")

## 6. 格式化数据集

In [ ]:
import json as _json
from datasets import Dataset

def load_and_format(filepath, tokenizer):
    """直接读 JSONL 并转为纯文本，绕过 datasets 的类型推断问题"""
    texts = []
    skipped = 0
    errors = []
    with open(filepath) as f:
        for i, line in enumerate(f):
            try:
                sample = _json.loads(line)
                messages = sample.get("messages", [])
                tools = sample.get("tools") or None
                # 清洗 messages，确保类型正确
                cleaned = []
                for m in messages:
                    msg = {"role": m["role"], "content": m.get("content")}
                    if m.get("tool_calls"):
                        tcs = []
                        for tc in m["tool_calls"]:
                            func = tc.get("function", tc)
                            args = func.get("arguments", {})
                            if isinstance(args, str):
                                try: args = _json.loads(args)
                                except: args = {}
                            if not isinstance(args, dict): args = {}
                            tcs.append({"type": "function", "function": {"name": func.get("name", ""), "arguments": args}})
                        msg["tool_calls"] = tcs
                        msg["content"] = None
                    if m.get("name"):
                        msg["name"] = m["name"]
                    cleaned.append(msg)
                text = tokenizer.apply_chat_template(
                    cleaned, tools=tools, tokenize=False, add_generation_prompt=False,
                )
                if text:
                    texts.append(text)
            except Exception as e:
                skipped += 1
                if len(errors) < 5:
                    errors.append(f"  Row {i}: {type(e).__name__}: {e}")
    print(f"  {filepath}: {len(texts)} OK, {skipped} skipped")
    if errors:
        print("  First errors:")
        for err in errors: print(err)
    return Dataset.from_dict({"text": texts})

print("格式化训练集...")
train_ds = load_and_format("/content/data/train.jsonl", tokenizer)
print("格式化验证集...")
valid_ds = load_and_format("/content/data/valid.jsonl", tokenizer)

print(f"\n训练集: {len(train_ds)} 样本")
print(f"验证集: {len(valid_ds)} 样本")

## 7. 开始训练

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch.fx.experimental._config as fx_config
fx_config.meta_nonzero_assume_all_nonzero = True

from unsloth import FastLanguageModel
FastLanguageModel.for_training(model)

# 优化器 CPU offload: optimizer states → CPU RAM，省 ~8GB 显存
import json as _j
ds_config = None
if vram_gb < 70:
    ds_config = '/content/ds_config.json'
    with open(ds_config, 'w') as f:
        _j.dump({
            'zero_optimization': {
                'stage': 2,
                'offload_optimizer': {'device': 'cpu', 'pin_memory': True},
                'allgather_partitions': True,
                'reduce_scatter': True,
            },
            'bf16': {'enabled': vram_gb >= 20},
            'fp16': {'enabled': vram_gb < 20},
            'train_batch_size': 'auto',
            'train_micro_batch_size_per_gpu': 'auto',
            'gradient_accumulation_steps': 'auto',
        }, f)
    print('DeepSpeed ZeRO-2 + CPU offload ON')

training_args = SFTConfig(
    output_dir="/content/output",
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    max_steps=1000,
    learning_rate=2e-5,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    bf16=vram_gb >= 20,
    fp16=vram_gb < 20,
    logging_steps=10,
    save_steps=1000,
    save_total_limit=1,
    seed=42,
    optim="adamw_8bit",
    max_seq_length=CONFIG["max_seq_length"],
    dataset_text_field="text",
    packing=True,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    torch_compile=True,    # 算子融合，省显存 + 加速 20-30%
    report_to="none",
    deepspeed=ds_config,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    train_dataset=train_ds,
    args=training_args,
)

print(f"有效 batch size: {CONFIG['batch_size'] * CONFIG['grad_accum']}")
print(f"总训练步数: 1000")
print("\n开始训练...\n")
print("\n开始训练...\n")

trainer.train()

## 8. 保存模型

In [ ]:
# 保存 LoRA 适配器
model.save_pretrained("/content/output/lora_adapter")
processor.save_pretrained("/content/output/lora_adapter")
print("LoRA 适配器已保存")

# 保存合并后的完整模型
model.save_pretrained_merged("/content/output/merged_bf16", processor, save_method="merged_16bit")
print("合并模型 (bf16) 已保存")

## 9. 测试工具调用

In [ ]:
FastLanguageModel.for_inference(model)

test_cases = [
    {
        "name": "中文-天气查询",
        "messages": [{"role": "user", "content": "请帮我查询北京今天的天气"}],
        "tools": [{"type": "function", "function": {
            "name": "get_weather", "description": "查询指定城市天气",
            "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
        }}],
    },
    {
        "name": "EN-File Search",
        "messages": [{"role": "user", "content": "Search for all Python files in the src directory"}],
        "tools": [{"type": "function", "function": {
            "name": "search_files", "description": "Search files by pattern",
            "parameters": {"type": "object", "properties": {"dir": {"type": "string"}, "pattern": {"type": "string"}}, "required": ["dir", "pattern"]}
        }}],
    },
    {
        "name": "中文-普通对话",
        "messages": [{"role": "user", "content": "你好，请介绍一下你自己"}],
        "tools": None,
    },
]

for tc in test_cases:
    print(f"\n{'='*50}")
    print(f"测试: {tc['name']}")
    print(f"User: {tc['messages'][0]['content']}")
    if tc.get('tools'): print(f"Tools: {[t['function']['name'] for t in tc['tools']]}")

    input_text = tokenizer.apply_chat_template(
        tc["messages"], tools=tc["tools"], tokenize=False, add_generation_prompt=True
    )
    inputs = processor(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, do_sample=True)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    if "<|im_end|>" in resp: resp = resp[:resp.index("<|im_end|>")]
    print(f"\nAssistant: {resp.strip()}")

## 10. 导出 GGUF (可选)

导出后可在本地用 Ollama / LM Studio 运行

In [ ]:
# 导出 GGUF Q4_K_M
model.save_pretrained_gguf(
    "/content/output/gguf",
    processor,
    quantization_method="q4_k_m",
)
print("GGUF 已导出!")

## 11. 保存到 Google Drive (防止 Colab 断连丢失)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/qwen35-tool-calling
!cp -r /content/output/lora_adapter /content/drive/MyDrive/qwen35-tool-calling/
!cp -r /content/output/gguf /content/drive/MyDrive/qwen35-tool-calling/ 2>/dev/null || true

print("模型已保存到 Google Drive: /MyDrive/qwen35-tool-calling/")

## 12. 推送到 HuggingFace Hub (可选)

In [ ]:
# HF_TOKEN = "hf_xxx"  # 取消注释并填入你的 token
# HF_REPO = "your-username/Qwen3.5-9B-Tool-Calling"  # 你的 repo

# model.push_to_hub_merged(HF_REPO, processor, save_method="merged_16bit", token=HF_TOKEN)
# print(f"已推送到: https://huggingface.co/{HF_REPO}")